# Financial Document NLP Pipeline
## Unstructured Text Analysis on CFPB Consumer Complaint Narratives

**Author:** Keith Tang  
**Data source:** [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) — public domain, US Government  
**Stack:** `requests` · `spaCy` · `transformers (FinBERT)` · `BERTopic` · `sentence-transformers` · `matplotlib` · `seaborn`

---

### What this notebook demonstrates

| Technique | Library | Relevance to financial services |
|-----------|---------|----------------------------------|
| Sentiment analysis | FinBERT (HuggingFace) | Score adviser reviews, complaint tone, disclosure sentiment |
| Named entity recognition | spaCy `en_core_web_lg` | Extract companies, products, risk terms from free text |
| Topic modelling | BERTopic | Discover latent complaint themes without labelled data |
| Compliance flag detection | sentence-transformers cosine similarity | Surface high-risk language patterns in narratives |
| Exploratory text analysis | pandas, matplotlib, wordcloud | Volume trends, issue breakdown, response-rate analysis |

---

## 0. Environment setup

In [ ]:
# Run once — comment out after first execution
!pip install -q transformers torch sentencepiece
!pip install -q bertopic sentence-transformers
!pip install -q spacy wordcloud umap-learn hdbscan kagglehub
!python -m spacy download en_core_web_lg -q

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

import spacy
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity

# Plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Environment ready.')

## 1. Data ingestion — CFPB Consumer Complaint Database (Kaggle)

We load the CFPB dataset via `kagglehub` — a pre-processed mirror of the official CFPB bulk export  
covering **2M+ complaints from 2011–2024**, filtered to rows with consumer narratives.  
Dataset: [`namigabbasov/consumer-complaint-dataset`](https://www.kaggle.com/datasets/namigabbasov/consumer-complaint-dataset)  
Source data: **US Government public domain** (CFPB Consumer Complaint Database).

> **Kaggle users:** the dataset is auto-downloaded via `kagglehub` with your session credentials.  
> **Colab users:** add your `kaggle.json` API key first — see the cell comment below.

In [ ]:
import kagglehub
import os

# Colab users: uncomment and run this block first
# from google.colab import files
# files.upload()  # upload your kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download dataset — cached after first run
print('Downloading CFPB dataset via kagglehub...')
path = kagglehub.dataset_download('namigabbasov/consumer-complaint-dataset')
print(f'Dataset path: {path}')

# Find the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'Files found: {csv_files}')
csv_path = os.path.join(path, csv_files[0])

# Load — dataset has 2M rows; sample for development speed
# Set FULL_LOAD = True to use the complete dataset (requires ~4GB RAM)
FULL_LOAD = False
LOAD_N = 50_000  # rows to sample for dev — increase as needed

print(f'Loading {"full" if FULL_LOAD else f"{LOAD_N:,} rows from"} dataset...')

if FULL_LOAD:
    raw_df = pd.read_csv(csv_path, low_memory=False)
else:
    # Read header to get columns, then sample
    raw_df = pd.read_csv(csv_path, nrows=LOAD_N, low_memory=False)

print(f'Raw shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head(2)

In [ ]:
# Print exact column names from the dataset so we can map correctly
print('Actual columns:')
for c in raw_df.columns:
    print(f'  {repr(c)}')


In [ ]:
# Robust column mapping — matches against lowercase stripped column names
# so it works regardless of capitalisation or spacing differences

WANTED = {
    'narrative':         ['narrative', 'consumer complaint narrative', 'consumer_complaint_narrative'],
    'product':           ['product_5', 'product'],
    'sub_product':       ['sub-product', 'sub_product'],
    'issue':             ['issue'],
    'company':           ['company'],
    'state':             ['state'],
    'company_response':  ['company response to consumer', 'company_response', 'company response'],
    'timely':            ['timely response?', 'timely response', 'timely'],
    'consumer_disputed': ['consumer disputed?', 'consumer disputed', 'consumer_disputed'],
    'date_received':     ['date received', 'date_received'],
    'complaint_id':      ['complaint id', 'complaint_id'],
}

# Build mapping from actual column → our standard name
actual_lower = {c.lower().strip(): c for c in raw_df.columns}
rename_map = {}
found = {}
missing = []

for std_name, candidates in WANTED.items():
    for cand in candidates:
        if cand.lower() in actual_lower:
            orig = actual_lower[cand.lower()]
            rename_map[orig] = std_name
            found[std_name] = orig
            break
    else:
        missing.append(std_name)

print('Mapped columns:')
for std, orig in found.items():
    print(f'  {orig!r:45s} -> {std}')
if missing:
    print(f'\nNot found (will be skipped): {missing}')

df = raw_df.rename(columns=rename_map).copy()

# Add placeholder columns for anything missing so downstream cells don't KeyError
for col in missing:
    df[col] = None

# Filter to rows with narratives
df = df.dropna(subset=['narrative'])
df['narrative'] = df['narrative'].astype(str).str.strip()
df = df[df['narrative'].str.len() > 50].reset_index(drop=True)

# Parse date if present
if df['date_received'].notna().any():
    df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')

print(f'\nComplaints with narratives: {len(df):,}')
print(f'Products: {df["product"].nunique()}')
print(df['product'].value_counts().head(8).to_string())
df.head(2)


## 2. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Volume by product ---
prod_counts = df['product'].value_counts().head(8)
axes[0].barh(prod_counts.index, prod_counts.values, color='#378ADD', edgecolor='white')
axes[0].set_title('Complaints by product')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# --- Top companies ---
top_companies = df['company'].value_counts().head(10)
axes[1].barh(top_companies.index, top_companies.values, color='#1D9E75', edgecolor='white')
axes[1].set_title('Top 10 companies by complaint volume')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

# --- Company response breakdown (skip if column is empty) ---
if df['company_response'].notna().any():
    resp = df['company_response'].value_counts().head(6)
    axes[2].bar(range(len(resp)), resp.values, color='#7F77DD', edgecolor='white')
    axes[2].set_xticks(range(len(resp)))
    axes[2].set_xticklabels(resp.index, rotation=30, ha='right', fontsize=8)
    axes[2].set_title('Company response types')
    axes[2].set_ylabel('Count')
else:
    # Fallback: issue breakdown
    issue_counts = df['issue'].value_counts().head(6)
    axes[2].barh(issue_counts.index, issue_counts.values, color='#7F77DD', edgecolor='white')
    axes[2].set_title('Top issues')
    axes[2].invert_yaxis()

plt.suptitle('CFPB Consumer Complaints — Overview', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('cfpb_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Narrative length distribution
df['narrative_len'] = df['narrative'].str.len()
df['word_count'] = df['narrative'].str.split().str.len()

print(df[['narrative_len', 'word_count']].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df['word_count'].clip(upper=500), bins=40, color='#378ADD', edgecolor='white')
axes[0].set_title('Word count distribution (capped at 500)')
axes[0].set_xlabel('Words per narrative')

# Wordcloud across all narratives
text_blob = ' '.join(df['narrative'].sample(300, random_state=SEED).tolist())
# Remove CFPB redaction placeholder
text_blob = re.sub(r'XX+', '', text_blob)
wc = WordCloud(width=600, height=300, background_color='white',
               colormap='Blues', max_words=80,
               stopwords={'XXXX', 'told', 'called', 'account', 'said'}).generate(text_blob)
axes[1].imshow(wc, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Narrative word cloud (sample of 300)')

plt.tight_layout()
plt.savefig('cfpb_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Sentiment analysis — FinBERT

`ProsusAI/finbert` is a BERT model fine-tuned on financial text.  
We score each narrative at sentence level and aggregate to document level.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

finbert_tokenizer = BertTokenizer.from_pretrained('ProsusAI/finbert')
finbert_model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert').to(DEVICE)
finbert_model.eval()

SENTIMENT_LABELS = ['positive', 'negative', 'neutral']  # FinBERT label order


def finbert_score(text: str, max_tokens: int = 512) -> dict:
    """Return {positive, negative, neutral} probability dict for a text snippet."""
    inputs = finbert_tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_tokens, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = finbert_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    return {lbl: round(float(p), 4) for lbl, p in zip(SENTIMENT_LABELS, probs)}


def score_narrative(narrative: str, max_chars: int = 800) -> dict:
    """Truncate to first max_chars for speed; score document-level."""
    text = re.sub(r'XX+', '[REDACTED]', narrative)[:max_chars]
    return finbert_score(text)


# Score a subset for speed (increase as compute allows)
SAMPLE_N = 300
sample_df = df.sample(n=SAMPLE_N, random_state=SEED).copy()

print(f'Scoring {SAMPLE_N} narratives with FinBERT...')
scores = []
for i, row in sample_df.iterrows():
    scores.append(score_narrative(row['narrative']))
    if (len(scores)) % 50 == 0:
        print(f'  {len(scores)}/{SAMPLE_N}')

sent_df = pd.DataFrame(scores)
sample_df = sample_df.reset_index(drop=True)
sample_df[['sent_positive', 'sent_negative', 'sent_neutral']] = sent_df.values
sample_df['sentiment'] = sent_df.idxmax(axis=1)
sample_df['confidence'] = sent_df.max(axis=1)

print('\nSentiment distribution:')
print(sample_df['sentiment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Sentiment by product ---
sent_prod = sample_df.groupby(['product', 'sentiment']).size().unstack(fill_value=0)
sent_prod_pct = sent_prod.div(sent_prod.sum(axis=1), axis=0) * 100
colors_map = {'positive': '#639922', 'neutral': '#888780', 'negative': '#E24B4A'}
sent_prod_pct[[c for c in ['positive', 'neutral', 'negative'] if c in sent_prod_pct.columns]].plot(
    kind='barh', stacked=True, ax=axes[0],
    color=[colors_map[c] for c in ['positive', 'neutral', 'negative'] if c in sent_prod_pct.columns],
    edgecolor='white'
)
axes[0].set_title('Sentiment by product (% of complaints)')
axes[0].set_xlabel('Percentage (%)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].invert_yaxis()

# --- Confidence distribution ---
for sent, color in colors_map.items():
    subset = sample_df[sample_df['sentiment'] == sent]['confidence']
    if len(subset) > 0:
        axes[1].hist(subset, bins=20, alpha=0.6, label=sent, color=color)
axes[1].set_title('FinBERT confidence score distribution')
axes[1].set_xlabel('Confidence (softmax probability)')
axes[1].legend()

plt.suptitle('FinBERT Sentiment Analysis — CFPB Narratives', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('cfpb_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSample high-confidence negative complaints:')
neg_examples = sample_df[
    (sample_df['sentiment'] == 'negative') & (sample_df['confidence'] > 0.85)
][['product', 'narrative', 'confidence']].head(3)
for _, row in neg_examples.iterrows():
    print(f"\n[{row['product']}] confidence={row['confidence']:.3f}")
    print(row['narrative'][:300], '...')

## 4. Named entity recognition — spaCy

We use spaCy's `en_core_web_lg` model to extract ORG, PRODUCT, MONEY, DATE, PERSON, and LAW entities.  
A custom rule layer flags domain-specific financial risk terms.

In [ ]:
nlp = spacy.load('en_core_web_lg')

# Custom risk term ruler — patterns added on top of the pre-trained NER
RISK_TERMS = [
    'foreclosure', 'default', 'bankruptcy', 'charge-off', 'late fee',
    'repossession', 'identity theft', 'unauthorized charge', 'overdraft',
    'interest rate increase', 'debt collector', 'credit report error',
    'unfair practice', 'robo-signed', 'predatory lending',
]

ruler = nlp.add_pipe('span_ruler', before='ner', config={'overwrite': False})
patterns = [
    {"label": "RISK_TERM", "pattern": term} for term in RISK_TERMS
]
ruler.add_patterns(patterns)


def extract_entities(text: str) -> list[tuple]:
    """Return list of (entity_text, label) tuples for standard + custom labels."""
    clean = re.sub(r'XX+', '[REDACTED]', text)[:1000]  # truncate for speed
    doc = nlp(clean)
    keep_labels = {'ORG', 'PRODUCT', 'MONEY', 'DATE', 'PERSON', 'LAW', 'RISK_TERM'}
    ents = [(ent.text.strip(), ent.label_) for ent in doc.ents if ent.label_ in keep_labels]
    # Also check span_ruler hits
    spans = [(span.text.strip(), span.label_) for span in doc.spans.get('ruler', [])]
    return list({(t, l) for t, l in ents + spans if len(t) > 1})


print('Running NER on sample...')
NER_SAMPLE = 150
ner_df = sample_df.head(NER_SAMPLE).copy()
ner_df['entities'] = ner_df['narrative'].apply(extract_entities)
ner_df['entity_count'] = ner_df['entities'].apply(len)

# Flatten all entities
all_ents = [(ent, lbl) for ents in ner_df['entities'] for ent, lbl in ents]
ent_df = pd.DataFrame(all_ents, columns=['entity', 'label'])

print(f'\nTotal entity mentions: {len(ent_df):,}')
print(ent_df['label'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Entity label distribution ---
label_counts = ent_df['label'].value_counts()
label_colors = {
    'ORG': '#378ADD', 'PRODUCT': '#1D9E75', 'RISK_TERM': '#E24B4A',
    'MONEY': '#BA7517', 'DATE': '#888780', 'PERSON': '#7F77DD', 'LAW': '#D85A30'
}
bar_colors = [label_colors.get(l, '#888780') for l in label_counts.index]
axes[0].bar(label_counts.index, label_counts.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Entity type distribution')
axes[0].set_ylabel('Count')

# --- Top ORG entities (companies most mentioned) ---
top_orgs = ent_df[ent_df['label'] == 'ORG']['entity'].value_counts().head(12)
axes[1].barh(top_orgs.index, top_orgs.values, color='#378ADD', edgecolor='white')
axes[1].set_title('Top 12 organisations mentioned in narratives')
axes[1].set_xlabel('Mention count')
axes[1].invert_yaxis()

plt.suptitle('spaCy Named Entity Recognition — CFPB Narratives', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('cfpb_ner.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Risk term frequency ---
risk_terms = ent_df[ent_df['label'] == 'RISK_TERM']['entity'].str.lower().value_counts()
print('\nTop risk terms detected:')
print(risk_terms.to_string())

## 5. Topic modelling — BERTopic

BERTopic uses sentence embeddings + UMAP dimensionality reduction + HDBSCAN clustering.  
No labels needed — topics emerge from the text itself.

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Use a cleaned corpus — remove redaction markers for embedding quality
TOPIC_N = 400
topic_corpus = (
    df['narrative']
    .sample(n=min(TOPIC_N, len(df)), random_state=SEED)
    .apply(lambda x: re.sub(r'XX+', '', x)[:600].strip())
    .tolist()
)

print(f'Fitting BERTopic on {len(topic_corpus)} documents...')

umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0,
    metric='cosine', random_state=SEED
)
hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5,
    metric='euclidean', cluster_selection_method='eom', prediction_data=True
)
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics='auto',
    verbose=True
)

topics, probs = topic_model.fit_transform(topic_corpus)

topic_info = topic_model.get_topic_info()
print(f'\nTopics discovered: {len(topic_info) - 1} (excluding outlier topic -1)')
print(topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']].head(10).to_string(index=False))

In [ ]:
# Top topics by document count
top_topics = topic_info[topic_info['Topic'] != -1].nlargest(8, 'Count')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Topic size bar chart ---
topic_colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(top_topics)))
axes[0].barh(
    [f"Topic {r['Topic']}: {r['Name'][:35]}" for _, r in top_topics.iterrows()],
    top_topics['Count'].values,
    color=topic_colors, edgecolor='white'
)
axes[0].set_title('Top 8 topics by document count')
axes[0].set_xlabel('Documents')
axes[0].invert_yaxis()

# --- Top keywords per top 4 topics ---
top4 = top_topics.head(4)['Topic'].tolist()
kw_data = []
for t in top4:
    words = topic_model.get_topic(t)
    if words:
        kw_data.append((t, [(w, s) for w, s in words[:8]]))

if kw_data:
    all_kw_words = [w for _, kws in kw_data for w, _ in kws]
    all_kw_scores = [s for _, kws in kw_data for _, s in kws]
    kw_colors = [plt.cm.Greens(0.4 + 0.5 * (i // 8) / max(len(kw_data)-1, 1))
                 for i in range(len(all_kw_words))]
    axes[1].barh(all_kw_words[::-1], all_kw_scores[::-1], color=kw_colors[::-1], edgecolor='white')
    axes[1].set_title('Top keyword scores — top 4 topics')
    axes[1].set_xlabel('c-TF-IDF score')

plt.suptitle('BERTopic — CFPB Complaint Theme Discovery', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('cfpb_topics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Compliance flag detection — embedding similarity

We define a set of regulatory risk phrase templates and use cosine similarity between sentence embeddings to surface complaints that semantically match — catching paraphrases that keyword search would miss.

In [ ]:
# Compliance risk phrases — modelled on CFPB enforcement priorities
COMPLIANCE_FLAGS = {
    'Undisclosed fees': [
        'charged fees that were not disclosed',
        'hidden fees added without notice',
        'unexpected charges not in the agreement',
    ],
    'Robo-signing / document fraud': [
        'signed documents I never agreed to',
        'forged signature on mortgage documents',
        'documents were signed without my knowledge',
    ],
    'Harassment by debt collector': [
        'called me multiple times every day',
        'threatened legal action over a debt',
        'debt collector contacted my employer',
    ],
    'Credit reporting error': [
        'incorrect information on my credit report',
        'account I never opened appeared on my credit file',
        'negative mark not removed after dispute',
    ],
    'Predatory / unfair lending': [
        'interest rate raised without proper notice',
        'loan terms changed after signing',
        'forced into a loan I could not afford',
    ],
    'Identity theft / unauthorized account': [
        'account opened without my permission',
        'someone used my personal information to apply for credit',
        'fraudulent account on my credit report',
    ],
}

# Embed all flag phrases once
flat_flags = [(cat, phrase) for cat, phrases in COMPLIANCE_FLAGS.items() for phrase in phrases]
flag_texts = [p for _, p in flat_flags]
flag_cats = [c for c, _ in flat_flags]
flag_embeddings = embedder.encode(flag_texts, show_progress_bar=False)

SIMILARITY_THRESHOLD = 0.55


def detect_flags(narrative: str) -> list[str]:
    """Return list of compliance flag categories triggered by this narrative."""
    clean = re.sub(r'XX+', '', narrative)[:600]
    emb = embedder.encode([clean])
    sims = cosine_similarity(emb, flag_embeddings)[0]
    triggered = set()
    for i, sim in enumerate(sims):
        if sim >= SIMILARITY_THRESHOLD:
            triggered.add(flag_cats[i])
    return sorted(triggered)


print('Running compliance flag detection...')
FLAG_N = 200
flag_sample = df.sample(n=min(FLAG_N, len(df)), random_state=SEED).copy().reset_index(drop=True)
flag_sample['flags'] = flag_sample['narrative'].apply(detect_flags)
flag_sample['n_flags'] = flag_sample['flags'].apply(len)
flag_sample['flagged'] = flag_sample['n_flags'] > 0

print(f'Complaints with at least one flag: {flag_sample["flagged"].sum()} / {FLAG_N}')
print(f'Flag rate: {flag_sample["flagged"].mean():.1%}')

In [ ]:
# Count by flag category
all_flags_list = [f for flags in flag_sample['flags'] for f in flags]
flag_counts = pd.Series(Counter(all_flags_list)).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

flag_colors = plt.cm.Oranges(np.linspace(0.4, 0.85, len(flag_counts)))
axes[0].barh(flag_counts.index, flag_counts.values, color=flag_colors, edgecolor='white')
axes[0].set_title('Compliance flags by category')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# --- Flags vs company response (only if column available) ---
if flag_sample['company_response'].notna().any():
    flag_vs_response = flag_sample.groupby(['flagged', 'company_response']).size().unstack(fill_value=0)
    flag_vs_response.index = ['Not flagged', 'Flagged']
    flag_vs_response.T.plot(kind='bar', ax=axes[1], color=['#888780', '#E24B4A'], edgecolor='white')
    axes[1].set_title('Company response type — flagged vs unflagged')
    axes[1].set_xlabel('')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right', fontsize=8)
    axes[1].legend(fontsize=9)
else:
    # Fallback: flag rate by product
    flag_by_prod = flag_sample.groupby('product')['flagged'].mean().sort_values(ascending=False)
    axes[1].barh(flag_by_prod.index, flag_by_prod.values * 100, color='#E24B4A', edgecolor='white')
    axes[1].set_title('Compliance flag rate by product (%)')
    axes[1].set_xlabel('Flag rate (%)')
    axes[1].invert_yaxis()

plt.suptitle('Compliance Flag Detection — Embedding Similarity Approach', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('cfpb_flags.png', dpi=150, bbox_inches='tight')
plt.show()

# Show flagged examples
print('\nSample flagged complaints:')
for _, row in flag_sample[flag_sample['flagged']].head(3).iterrows():
    print(f"\nFlags: {row['flags']}")
    print(row['narrative'][:250], '...')


## 7. Summary — aggregate metrics table

In [ ]:
sentiment_dist = sample_df['sentiment'].value_counts()

summary = {
    'Total complaints fetched': len(df),
    'Complaints with narratives': len(df),
    'FinBERT sample size': SAMPLE_N,
    'Negative sentiment (%)': f"{sentiment_dist.get('negative', 0) / SAMPLE_N:.1%}",
    'Neutral sentiment (%)': f"{sentiment_dist.get('neutral', 0) / SAMPLE_N:.1%}",
    'Positive sentiment (%)': f"{sentiment_dist.get('positive', 0) / SAMPLE_N:.1%}",
    'Mean FinBERT confidence': f"{sample_df['confidence'].mean():.3f}",
    'NER sample size': NER_SAMPLE,
    'Total entity mentions (NER sample)': len(ent_df),
    'Risk terms flagged (NER)': int((ent_df['label'] == 'RISK_TERM').sum()),
    'BERTopic corpus size': len(topic_corpus),
    'Topics discovered': int(len(topic_info[topic_info['Topic'] != -1])),
    'Compliance flag sample size': FLAG_N,
    'Complaints with compliance flag': int(flag_sample['flagged'].sum()),
    'Compliance flag rate': f"{flag_sample['flagged'].mean():.1%}",
    'Similarity threshold': SIMILARITY_THRESHOLD,
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
summary_df.index.name = 'Metric'
print('='*55)
print('CFPB NLP PIPELINE — RESULTS SUMMARY')
print('='*55)
print(summary_df.to_string())
print('='*55)

## 8. Next steps

| Extension | Description |
|-----------|-------------|
| Fine-tune FinBERT | Label ~500 CFPB narratives and fine-tune for financial complaint severity |
| LLM summarisation | Use `claude-sonnet` or `gpt-4o` to generate structured summaries of each complaint cluster |
| Streamlit dashboard | Deploy the pipeline as an interactive app — upload a batch of complaint CSVs and get flagged output |
| Time-series analysis | Track complaint volume and flag rates by company over rolling 90-day windows |
| Cross-lingual NER | Apply `xlm-roberta` to handle multilingual complaint datasets |

---

**Data:** CFPB Consumer Complaint Database — public domain, US Government  
**Repository:** [github.com/keithtang/cfpb-nlp-pipeline](https://github.com/keithtang/cfpb-nlp-pipeline)  
**Contact:** [LinkedIn](https://linkedin.com/in/keithtang)